# Demo: Vietnamese Fake News Detection

Runs the trained TF-IDF + Logistic Regression pipeline end-to-end on a few sample inputs and visualises confusion matrices on the saved test split.

Run from the project root after `python src/train.py` has produced the model artifacts:

```bash
pip install jupyter matplotlib
jupyter notebook notebooks/demo.ipynb
```

In [ ]:
import sys
from pathlib import Path

# Allow imports from the project root when running inside notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import json
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

MODEL_PATH = ROOT / 'models' / 'logistic_regression.pkl'
METRICS_PATH = ROOT / 'models' / 'models_metrics.json'
TEST_CSV = ROOT / 'data' / 'processed' / 'test_clean.csv'
THRESHOLD_PATH = ROOT / 'models' / 'threshold.json'

model = joblib.load(MODEL_PATH)
metrics = json.loads(METRICS_PATH.read_text(encoding='utf-8'))
threshold = json.loads(THRESHOLD_PATH.read_text(encoding='utf-8'))['fake_threshold']
print(f'Loaded {type(model).__name__} with threshold = {threshold:.2f}')
metrics

## 1. Predict on a few sample texts

Uses `detector.services.predict_news` so the demo matches what the Django app sees.

In [ ]:
from detector.services import predict_news

samples = [
    'Bộ Y tế khuyến cáo người dân tiêm vắc xin phòng cúm mùa định kỳ.',
    'BÍ MẬT: Uống 1 cốc nước chanh mỗi sáng chữa khỏi mọi loại ung thư trong 7 ngày!',
    'Theo Tổng cục Thống kê, GDP quý 1 năm nay tăng 5,66% so với cùng kỳ.',
    'Tin nóng: Cơ hội đầu tư tiền ảo lãi suất 300% mỗi tháng, không rủi ro, đăng ký ngay!',
]

rows = []
for s in samples:
    r = predict_news(s)
    rows.append({
        'text': s[:80] + ('...' if len(s) > 80 else ''),
        'label': r['prediction'],
        'fake_%': r['fake_probability'],
        'top_fake': ', '.join(f"{f['term']} (+{f['weight']:.2f})" for f in r['top_fake_features'][:3]),
    })

pd.DataFrame(rows)

## 2. Confusion matrix on the held-out test set

In [ ]:
test_df = pd.read_csv(TEST_CSV)
probs = model.predict_proba(test_df['cleaned_text'])[:, list(model.classes_).index(1)]
preds_default = (probs >= 0.5).astype(int)
preds_tuned = (probs >= threshold).astype(int)

for name, preds in [('default 0.5', preds_default), (f'tuned {threshold:.2f}', preds_tuned)]:
    print(f'--- Threshold {name} ---')
    print(classification_report(test_df['label'], preds, target_names=['Real', 'Fake'], digits=4))
    print(pd.DataFrame(confusion_matrix(test_df['label'], preds), index=['true Real', 'true Fake'], columns=['pred Real', 'pred Fake']))
    print()

## 3. Top global features per class

Inspects the LR coefficients directly — these are the same weights that power per-input explanations in the web UI.

In [ ]:
vectorizer = model.named_steps['tfidf']
classifier = model.named_steps['classifier']
feature_names = vectorizer.get_feature_names_out()
coef = np.asarray(classifier.coef_).ravel()

top_fake_idx = np.argsort(coef)[::-1][:15]
top_real_idx = np.argsort(coef)[:15]

pd.DataFrame({
    'fake_indicator': [feature_names[i] for i in top_fake_idx],
    'fake_weight': [round(float(coef[i]), 4) for i in top_fake_idx],
    'real_indicator': [feature_names[i] for i in top_real_idx],
    'real_weight': [round(float(coef[i]), 4) for i in top_real_idx],
})